The MIT License (MIT)

Copyright (c) 2021 NVIDIA CORPORATION

Permission is hereby granted, free of charge, to any person obtaining a copy of
this software and associated documentation files (the "Software"), to deal in
the Software without restriction, including without limitation the rights to
use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of
the Software, and to permit persons to whom the Software is furnished to do so,
subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS
FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR
COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER
IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN
CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

In [2]:
# !pip install wheel sebtuptools --upgrade install cupy-cuda11x


In [ ]:
# !pip install wheel setuptools --upgrade install cudf-cu11 --extra-index-url=https://pypi.ngc.nvidia.com install cupy-cuda11xA
!
chmod -v +x Miniconda*.sh
!wget https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
!chmod -v +x Miniconda*.sh
!echo "HASH_OF_INSTALLER *INSTALLER_SCRIPT_NAME" | shasum --check
!echo "78f39f9bae971ec1ae7969f0516017f2413f17796670f7040725dd83fcff5689 *Miniconda3-py39_4.12.0-Linux-x86_64.sh" | shasum --check
!echo "78f39f9bae971ec1ae7969f0516017f2413f17796670f7040725dd83fcff5689 *Miniconda3-py39_4.12.0-Linux-x86_64.sh" | shasum --check
!echo "78f39f9bae971ec1ae7969f0516017f2413f17796670f7040725dd83fcff5689 *Miniconda3-py39_4.12.0-Linux-x86_64.sh" | shasum --check
!./Miniconda3-py39_4.12.0-Linux-x86_64.sh
!if ! [[ $PATH =~ "$HOME/miniconda3/bin" ]]; then
!  PATH="$HOME/miniconda3/bin:$PATH"
!fi
!conda init bash
!conda init zsh
!source ~/.bashrc  # Bash kullanıyorsanız
!source ~/.zshrc   # Zsh kullanıyorsanız

mode of 'Miniconda3-py39_4.12.0-Linux-x86_64.sh' retained as 0755 (rwxr-xr-x)
--2023-11-15 18:35:59--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.131.3, 104.16.130.3, 2606:4700::6810:8303, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.131.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76607678 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh.1’

Miniconda3-py39_4.1 100%[===================>]  73.06M   139MB/s    in 0.5s    

2023-11-15 18:36:00 (139 MB/s) - ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh.1’ saved [76607678/76607678]

mode of 'Miniconda3-py39_4.12.0-Linux-x86_64.sh' retained as 0755 (rwxr-xr-x)
shasum: standard input: no properly formatted SHA checksum lines found
Miniconda3-py39_4.12.0-Linux-x86_64.sh: OK
Miniconda3-py39_4.12.0-Linux-x86_64.sh: OK
Miniconda3-py39_4.12.0-Linux-x86_64.sh: OK

Welcome to Miniconda3 py39

In [1]:
import pandas as pd
import cudf
from sklearn.model_selection import GroupKFold

pd.__version__, cudf.__version__

ModuleNotFoundError: No module named 'pandas'

In [71]:
from numba import cuda

def get_order_in_group(utrip_id_,order):
    for i in range(cuda.threadIdx.x, len(utrip_id_), cuda.blockDim.x):
        order[i] = i

def add_cumcount(df, sort_col, outputname):
    df = df.sort_values(sort_col, ascending=True)
    tmp = df[['utrip_id_', 'checkin']].groupby(['utrip_id_']).apply_grouped(
        get_order_in_group,incols=['utrip_id_'],
        outcols={'order': 'int32'},
        tpb=32)
    tmp.columns = ['utrip_id_', 'checkin', outputname]
    df = df.merge(tmp, how='left', on=['utrip_id_', 'checkin'])
    df = df.sort_values(sort_col, ascending=True)
    return(df)

In [72]:
train = cudf.read_csv('../00_Data/train_set.csv').sort_values(by=['user_id','checkin'])
# train = cudf.read_csv('../00_Data/booking_train_set.csv').sort_values(by=['user_id','checkin'])

test = cudf.read_csv('../00_Data/test_set.csv').sort_values(by=['user_id','checkin'])
# test = cudf.read_csv('../00_Data/booking_test_set.csv').sort_values(by=['user_id','checkin'])
# del train['Unnamed: 0']
# del test['row_num'], test['total_rows']

print(train.shape, test.shape)
train.head()

(1166835, 9) (378667, 9)


,user_id,checkin,checkout,city_id,device_class,affiliate_id,booker_country,hotel_country,utrip_id
413669,29,2016-07-09,2016-07-11,47054,desktop,1601,Elbonia,Elbonia,29_1
413670,29,2016-07-11,2016-07-13,34444,desktop,1601,Elbonia,Elbonia,29_1
413671,29,2016-07-13,2016-07-16,12291,desktop,1601,Elbonia,Elbonia,29_1
413672,29,2016-07-16,2016-07-18,16386,desktop,8132,Elbonia,Elbonia,29_1
1128910,81,2016-05-15,2016-05-16,33665,desktop,9924,Elbonia,Elbonia,81_1


In [73]:
train[train['user_id'] == 29 ]

,user_id,checkin,checkout,city_id,device_class,affiliate_id,booker_country,hotel_country,utrip_id
413669,29,2016-07-09,2016-07-11,47054,desktop,1601,Elbonia,Elbonia,29_1
413670,29,2016-07-11,2016-07-13,34444,desktop,1601,Elbonia,Elbonia,29_1
413671,29,2016-07-13,2016-07-16,12291,desktop,1601,Elbonia,Elbonia,29_1
413672,29,2016-07-16,2016-07-18,16386,desktop,8132,Elbonia,Elbonia,29_1


In [74]:
test[test['user_id'] == 29 ]

,user_id,checkin,checkout,device_class,affiliate_id,booker_country,utrip_id,city_id,hotel_country


In [75]:
train['istest'] = 0
test['istest'] = 1
raw = cudf.concat([train,test], sort=False )
raw = raw.sort_values( ['user_id','checkin'], ascending=True )
raw.head()

,user_id,checkin,checkout,city_id,device_class,affiliate_id,booker_country,hotel_country,utrip_id,istest
413669,29,2016-07-09,2016-07-11,47054,desktop,1601,Elbonia,Elbonia,29_1,0
413670,29,2016-07-11,2016-07-13,34444,desktop,1601,Elbonia,Elbonia,29_1,0
413671,29,2016-07-13,2016-07-16,12291,desktop,1601,Elbonia,Elbonia,29_1,0
413672,29,2016-07-16,2016-07-18,16386,desktop,8132,Elbonia,Elbonia,29_1,0
355509,65,2016-09-26,2016-09-29,36403,desktop,3577,The Devilfire Empire,Cobra Island,65_1,1


In [76]:
raw['fold'] = 0
group_kfold = GroupKFold(n_splits=5)
for fold, (train_index, test_index) in enumerate(group_kfold.split(X=raw, y=raw, groups=raw['utrip_id'].to_pandas())):
    raw.iloc[test_index,10] = fold

raw['fold'].value_counts()

1    309101
0    309101
3    309100
2    309100
4    309100
Name: fold, dtype: int32

In [77]:
#This flag tell which row must be part of the submission file.

raw['submission'] = 0
raw.loc[ (raw.city_id==0)&(raw.istest) ,'submission'] = 1

raw.loc[ raw.submission==1 ]

,user_id,checkin,checkout,city_id,device_class,affiliate_id,booker_country,hotel_country,utrip_id,istest,fold,submission
355513,65,2016-10-03,2016-10-04,0,mobile,4132,The Devilfire Empire,<NA>,65_1,1,2,1
356899,67,2016-08-11,2016-08-14,0,desktop,9924,Tcherkistan,<NA>,67_1,1,1,1
10963,115,2016-04-06,2016-04-07,0,desktop,9924,Elbonia,<NA>,115_1,1,0,1
120565,279,2016-03-27,2016-04-01,0,desktop,2803,Tcherkistan,<NA>,279_1,1,3,1
139366,307,2016-06-02,2016-06-03,0,desktop,8132,Elbonia,<NA>,307_1,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
353170,6257974,2016-07-05,2016-07-07,0,tablet,7974,The Devilfire Empire,<NA>,6257974_1,1,2,1
353174,6258010,2016-06-10,2016-06-12,0,mobile,359,The Devilfire Empire,<NA>,6258010_1,1,0,1
353181,6258104,2016-08-25,2016-08-27,0,mobile,359,Gondal,<NA>,6258104_4,1,1,1
353186,6258120,2016-07-24,2016-07-25,0,desktop,9924,Gondal,<NA>,6258120_1,1,4,1


In [78]:
#number of places visited in each trip

aggs = raw.groupby('utrip_id', as_index=False)['user_id'].count().reset_index()
aggs.columns = ['utrip_id', 'N']
raw = raw.merge(aggs, on=['utrip_id'], how='inner')

In [79]:
raw['utrip_id_'], mp = raw['utrip_id'].factorize()

In [80]:
raw = add_cumcount(raw, ['utrip_id_','checkin'], 'dcount')

In [81]:
raw['icount'] = raw['N']-raw['dcount']-1

In [82]:
raw.head(50)

,user_id,checkin,checkout,city_id,device_class,affiliate_id,booker_country,hotel_country,utrip_id,istest,fold,submission,N,utrip_id_,dcount,icount
10368,1000027,2016-08-13,2016-08-14,8183,desktop,7168,Elbonia,Gondal,1000027_1,0,0,0,4,0,0,3
10369,1000027,2016-08-14,2016-08-16,15626,desktop,7168,Elbonia,Gondal,1000027_1,0,0,0,4,0,1,2
10370,1000027,2016-08-16,2016-08-18,60902,desktop,7168,Elbonia,Gondal,1000027_1,0,0,0,4,0,2,1
10371,1000027,2016-08-18,2016-08-21,30628,desktop,253,Elbonia,Gondal,1000027_1,0,0,0,4,0,3,0
10372,1000033,2016-04-09,2016-04-11,38677,mobile,359,Gondal,Cobra Island,1000033_1,0,0,0,5,1,0,4
10373,1000033,2016-04-11,2016-04-12,52089,desktop,384,Gondal,Cobra Island,1000033_1,0,0,0,5,1,1,3
10374,1000033,2016-04-12,2016-04-14,21328,desktop,384,Gondal,Cobra Island,1000033_1,0,0,0,5,1,2,2
10375,1000033,2016-04-14,2016-04-16,27485,desktop,384,Gondal,Cobra Island,1000033_1,0,0,0,5,1,3,1
10376,1000033,2016-04-16,2016-04-19,38677,desktop,384,Gondal,Cobra Island,1000033_1,0,0,0,5,1,4,0
10377,1000045,2016-06-18,2016-06-20,64876,desktop,2790,The Devilfire Empire,Fook Island,1000045_1,0,0,0,7,2,0,6


In [83]:
raw.to_csv( '../00_Data/train_and_test_2.csv', index=False )